# Neural Network Hyperparameter Sweeps - Weighted Season Average

This notebook runs W&B sweeps to find optimal hyperparameters for neural network models.

In [ ]:
%pip install dotenv wandb xgboost catboost lightning

In [1]:
import sys

sys.path.append("..")

import dotenv

import wandb
from src.api.run.neural_network import sweep_neural_network
from src.api.sweep import wandb_sweep

c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: v8-luky (aicomp-mmlm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Sweep 1: MSE Loss

In [3]:
max_runs = 100
sweep_config = {
    "name": "Neural Network (Weighted Average)",
    "method": "bayes",
    "metric": {"name": "val_brier", "goal": "minimize"},
    "parameters": {
        "neural_network_config": {
            "parameters": {
                "hidden_layers": {
                    "values": [
                        [64, 32],
                        [128, 64],
                        [128, 64, 32],
                        [256, 128],
                        [256, 128, 64],
                        [256, 128, 64, 32],
                        [512, 256],
                        [512, 256, 128],
                        [512, 256, 128, 64],
                        [512, 256, 128, 64, 32],
                        [512, 1024, 512, 256, 128],
                        [512, 1024, 512, 256, 128, 64],
                        [512, 1024, 512, 256, 128, 64, 32],
                    ]
                },
                "activation": {"values": ["relu", "leaky_relu", "gelu", "elu"]},
                "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "batch_norm": {"value": True},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.0001, "max": 0.01},
                "optimizer": {"value": "adamw"},
                "weight_decay": {"distribution": "uniform", "min": 0.0, "max": 0.1},
                "early_stopping_patience": {"value": 10},
                "seed": {"value": 42},
                "loss_function": {"value": "mse"},
                "scheduler": {"values": [None, "step", "cosine", "exponential", "reduce_on_plateau"]},
                "scheduler_step_size": {"values": [10, 15, 20, 25, 30]},
                "scheduler_gamma": {"distribution": "log_uniform_values", "min": 0.7, "max": 0.99},
                "scheduler_patience": {"values": [3, 5, 7]},
                "num_workers": {"value": 16},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 0},
                "start_season": {"value": 0},
                "num_features": {"distribution": "int_uniform", "min": 10, "max": 100},
                "data_loader": {"value": "weighted_season_average"},
                "data_loader_config": {
                    "parameters": {
                        "regular_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "tourney_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "discount_factor": {"distribution": "uniform", "min": 0.9, "max": 1.0},
                    }
                },
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_neural_network, run_count=max_runs, project="neural-network")

### Create Submission from Best Model

In [4]:
from src.dataloaders.simple import WeightedSeasonAvgDataLoader
from src.experiments import WandbTracker
from src.experiments.config import ExperimentConfig, RunConfig
from src.models.neural_network import NeuralNetworkHyperparamConfig, NeuralNetworkModel
from src.submissions import create_submission

In [5]:
run = wandb.Api().run("aicomp-mmlm/neural-network/mrc96iou")
config = run.config
config

{'config': "NeuralNetworkHyperparamConfig(hidden_layers=[64, 32], activation='gelu', dropout=0.3591515261148852, batch_norm=True, input_dropout=0.0, learning_rate=0.002633551270836464, batch_size=512, max_epochs=200, optimizer='adamw', weight_decay=0.03873843391709941, momentum=0.9, scheduler='exponential', scheduler_step_size=25, scheduler_gamma=0.9477039164797748, scheduler_patience=3, early_stopping=True, early_stopping_patience=10, early_stopping_min_delta=0.0001, loss_function='mse', validation_split=0.1, weight_init='xavier_uniform', gradient_clip_val=None, accumulate_grad_batches=1, num_workers=16, seed=42)",
 'input_dim': 85,
 'run_config': {'data_loader': 'weighted_season_average',
  'num_features': 89,
  'start_season': 0,
  'valid_season': 0,
  'data_loader_config': {'regular_weight': 0.9943383083708116,
   'tourney_weight': 0.9826034734482154,
   'discount_factor': 0.9539002462198868}},
 'neural_network_config': {'seed': 42,
  'dropout': 0.3591515261148852,
  'optimizer': '

In [6]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = WeightedSeasonAvgDataLoader(run_config.num_features, **run_config.data_loader_config)
run_config

RunConfig(num_features=89, valid_season=0, start_season=0, data_loader='weighted_season_average', data_loader_config={'regular_weight': 0.9943383083708116, 'tourney_weight': 0.9826034734482154, 'discount_factor': 0.9539002462198868})

In [7]:
hyperparameters = NeuralNetworkHyperparamConfig(**config.get("neural_network_config", {}))
hyperparameters

NeuralNetworkHyperparamConfig(hidden_layers=[64, 32], activation='gelu', dropout=0.3591515261148852, batch_norm=True, input_dropout=0.0, learning_rate=0.002633551270836464, batch_size=512, max_epochs=200, optimizer='adamw', weight_decay=0.03873843391709941, momentum=0.9, scheduler='exponential', scheduler_step_size=25, scheduler_gamma=0.9477039164797748, scheduler_patience=3, early_stopping=True, early_stopping_patience=10, early_stopping_min_delta=0.0001, loss_function='mse', validation_split=0.1, weight_init='xavier_uniform', gradient_clip_val=None, accumulate_grad_batches=1, num_workers=16, seed=42)

In [8]:
model = NeuralNetworkModel(
    dataloader,
    hyperparameters,
    None,
    WandbTracker(ExperimentConfig(project="neural-network", name="neural-network-standard"), run=run),
)

Seed set to 42


In [9]:
model.prepare_for_evaluation(
    "aicomp-mmlm/neural-network/model-mrc96iou:v2",
    valid_season=run_config.valid_season,
    start_season=run_config.start_season,
)

wandb:   1 of 1 files downloaded.  


Model loaded from checkpoint: c:\Users\kybur\Repos\HSLU\aicomp\code\notebooks\artifacts\model-mrc96iou-v2/model.ckpt
Scaler fitted on training data with 304299 samples and 85 features


In [10]:
season = 2025
create_submission(
    season=season, model=model, filename=f"submission_neural_network_weighted_average_{season}.csv", fit=False
)

WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/submission_neural_network_weighted_average_2025.csv')

## Sweep 2: BCE Loss

In [ ]:
max_runs = 100
sweep_config = {
    "name": "Neural Network (Weighted Average, BCE)",
    "method": "bayes",
    "metric": {"name": "val_brier", "goal": "minimize"},
    "parameters": {
        "neural_network_config": {
            "parameters": {
                "hidden_layers": {
                    "values": [
                        [64, 32],
                        [128, 64],
                        [128, 64, 32],
                        [256, 128],
                        [256, 128, 64],
                        [256, 128, 64, 32],
                        [512, 256],
                        [512, 256, 128],
                        [512, 256, 128, 64],
                        [512, 256, 128, 64, 32],
                        [512, 1024, 512, 256, 128],
                        [512, 1024, 512, 256, 128, 64],
                        [512, 1024, 512, 256, 128, 64, 32],
                    ]
                },
                "activation": {"values": ["relu", "leaky_relu", "gelu", "elu"]},
                "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "batch_norm": {"value": True},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.0001, "max": 0.01},
                "optimizer": {"value": "adamw"},
                "weight_decay": {"distribution": "uniform", "min": 0.0, "max": 0.1},
                "early_stopping_patience": {"value": 5},
                "seed": {"value": 42},
                "loss_function": {"value": "bce"},
                "scheduler": {"values": [None, "step", "cosine", "exponential", "reduce_on_plateau"]},
                "scheduler_step_size": {"values": [10, 15, 20, 25, 30]},
                "scheduler_gamma": {"distribution": "log_uniform_values", "min": 0.7, "max": 0.99},
                "scheduler_patience": {"values": [3, 5, 7]},
                "num_workers": {"value": 16},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 0},
                "start_season": {"value": 0},
                "num_features": {"distribution": "int_uniform", "min": 10, "max": 100},
                "data_loader": {"value": "weighted_season_average"},
                "data_loader_config": {
                    "parameters": {
                        "regular_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "tourney_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "discount_factor": {"distribution": "uniform", "min": 0.9, "max": 1.0},
                    }
                },
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_neural_network, run_count=max_runs, project="neural-network")

### Create Submission from Best Model

In [ ]:
from src.dataloaders.simple import WeightedSeasonAvgDataLoader
from src.experiments import WandbTracker
from src.experiments.config import ExperimentConfig, RunConfig
from src.models.neural_network import NeuralNetworkHyperparamConfig, NeuralNetworkModel
from src.submissions import create_submission

In [ ]:
run = wandb.Api().run("aicomp-mmlm/neural-network/mrc96iou")
config = run.config
config

{'config': "NeuralNetworkHyperparamConfig(hidden_layers=[64, 32], activation='gelu', dropout=0.3591515261148852, batch_norm=True, input_dropout=0.0, learning_rate=0.002633551270836464, batch_size=512, max_epochs=200, optimizer='adamw', weight_decay=0.03873843391709941, momentum=0.9, scheduler='exponential', scheduler_step_size=25, scheduler_gamma=0.9477039164797748, scheduler_patience=3, early_stopping=True, early_stopping_patience=10, early_stopping_min_delta=0.0001, loss_function='mse', validation_split=0.1, weight_init='xavier_uniform', gradient_clip_val=None, accumulate_grad_batches=1, num_workers=16, seed=42)",
 'input_dim': 85,
 'run_config': {'data_loader': 'weighted_season_average',
  'num_features': 89,
  'start_season': 0,
  'valid_season': 0,
  'data_loader_config': {'regular_weight': 0.9943383083708116,
   'tourney_weight': 0.9826034734482154,
   'discount_factor': 0.9539002462198868}},
 'neural_network_config': {'seed': 42,
  'dropout': 0.3591515261148852,
  'optimizer': '

In [ ]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = WeightedSeasonAvgDataLoader(run_config.num_features, **run_config.data_loader_config)
run_config

RunConfig(num_features=89, valid_season=0, start_season=0, data_loader='weighted_season_average', data_loader_config={'regular_weight': 0.9943383083708116, 'tourney_weight': 0.9826034734482154, 'discount_factor': 0.9539002462198868})

In [ ]:
hyperparameters = NeuralNetworkHyperparamConfig(**config.get("neural_network_config", {}))
hyperparameters

NeuralNetworkHyperparamConfig(hidden_layers=[64, 32], activation='gelu', dropout=0.3591515261148852, batch_norm=True, input_dropout=0.0, learning_rate=0.002633551270836464, batch_size=512, max_epochs=200, optimizer='adamw', weight_decay=0.03873843391709941, momentum=0.9, scheduler='exponential', scheduler_step_size=25, scheduler_gamma=0.9477039164797748, scheduler_patience=3, early_stopping=True, early_stopping_patience=10, early_stopping_min_delta=0.0001, loss_function='mse', validation_split=0.1, weight_init='xavier_uniform', gradient_clip_val=None, accumulate_grad_batches=1, num_workers=16, seed=42)

In [ ]:
model = NeuralNetworkModel(
    dataloader,
    hyperparameters,
    None,
    WandbTracker(ExperimentConfig(project="neural-network", name="neural-network-standard"), run=run),
)

Seed set to 42


In [ ]:
model.prepare_for_evaluation(
    "aicomp-mmlm/neural-network/model-mrc96iou:v2",
    valid_season=run_config.valid_season,
    start_season=run_config.start_season,
)

wandb:   1 of 1 files downloaded.  


Model loaded from checkpoint: c:\Users\kybur\Repos\HSLU\aicomp\code\notebooks\artifacts\model-mrc96iou-v2/model.ckpt
Scaler fitted on training data with 304299 samples and 85 features


In [ ]:
season = 2025
create_submission(
    season=season, model=model, filename=f"submission_neural_network_weighted_average_{season}.csv", fit=False
)

WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/submission_neural_network_weighted_average_2025.csv')

## Sweep 3: Default Features (num_features=0)

In [ ]:
max_runs = 100
sweep_config = {
    "name": "Neural Network (Weighted Average, Default Features)",
    "method": "bayes",
    "metric": {"name": "val_brier", "goal": "minimize"},
    "parameters": {
        "neural_network_config": {
            "parameters": {
                "hidden_layers": {
                    "values": [
                        [64, 32],
                        [128, 64],
                        [128, 64, 32],
                        [256, 128],
                        [256, 128, 64],
                        [256, 128, 64, 32],
                        [512, 256],
                        [512, 256, 128],
                        [512, 256, 128, 64],
                        [512, 256, 128, 64, 32],
                        [512, 1024, 512, 256, 128],
                        [512, 1024, 512, 256, 128, 64],
                        [512, 1024, 512, 256, 128, 64, 32],
                    ]
                },
                "activation": {"values": ["relu", "leaky_relu", "gelu", "elu"]},
                "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "batch_norm": {"value": True},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.0001, "max": 0.01},
                "optimizer": {"value": "adamw"},
                "weight_decay": {"distribution": "uniform", "min": 0.0, "max": 0.1},
                "early_stopping_patience": {"value": 10},
                "seed": {"value": 42},
                "loss_function": {"value": "mse"},
                "scheduler": {"values": [None, "step", "cosine", "exponential", "reduce_on_plateau"]},
                "scheduler_step_size": {"values": [10, 15, 20, 25, 30]},
                "scheduler_gamma": {"distribution": "log_uniform_values", "min": 0.7, "max": 0.99},
                "scheduler_patience": {"values": [3, 5, 7]},
                "num_workers": {"value": 16},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 0},
                "start_season": {"value": 0},
                "num_features": {"value": 0},
                "data_loader": {"value": "weighted_season_average"},
                "data_loader_config": {
                    "parameters": {
                        "regular_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "tourney_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "discount_factor": {"distribution": "uniform", "min": 0.9, "max": 1.0},
                    }
                },
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_neural_network, run_count=max_runs, project="neural-network")

## Sweep 4: BCE with Entropy Loss

In [ ]:
max_runs = 100
sweep_config = {
    "name": "Neural Network (Weighted Average, BCEwEntropy)",
    "method": "bayes",
    "metric": {"name": "val_brier", "goal": "minimize"},
    "parameters": {
        "neural_network_config": {
            "parameters": {
                "hidden_layers": {
                    "values": [
                        [64, 32],
                        [128, 64],
                        [128, 64, 32],
                        [256, 128],
                        [256, 128, 64],
                        [256, 128, 64, 32],
                        [512, 256],
                        [512, 256, 128],
                        [512, 256, 128, 64],
                        [512, 256, 128, 64, 32],
                        [512, 1024, 512, 256, 128],
                        [512, 1024, 512, 256, 128, 64],
                        [512, 1024, 512, 256, 128, 64, 32],
                    ]
                },
                "activation": {"values": ["relu", "leaky_relu", "gelu", "elu"]},
                "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "batch_norm": {"value": True},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.0001, "max": 0.01},
                "optimizer": {"value": "adamw"},
                "weight_decay": {"distribution": "uniform", "min": 0.0, "max": 0.1},
                "early_stopping_patience": {"value": 10},
                "seed": {"value": 42},
                "loss_function": {"value": "bce_entropy"},
                "scheduler": {"values": [None, "step", "cosine", "exponential", "reduce_on_plateau"]},
                "scheduler_step_size": {"values": [10, 15, 20, 25, 30]},
                "scheduler_gamma": {"distribution": "log_uniform_values", "min": 0.7, "max": 0.99},
                "scheduler_patience": {"values": [3, 5, 7]},
                "num_workers": {"value": 16},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 0},
                "start_season": {"value": 0},
                "num_features": {"distribution": "int_uniform", "min": 10, "max": 100},
                "data_loader": {"value": "weighted_season_average"},
                "data_loader_config": {
                    "parameters": {
                        "regular_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "tourney_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "discount_factor": {"distribution": "uniform", "min": 0.9, "max": 1.0},
                    }
                },
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_neural_network, run_count=max_runs, project="neural-network")